# Week 7 homework — Cellpose instance segmentation

**Machine Learning for Bioinformatics**

This notebook implements homework item 5 from Practical 7, Part VI:

> For Cellpose, compare at least two parameter settings involving model, flow threshold, cell-probability threshold, or minimum cell size. Show the masks and explain which setting is biologically more plausible.

It follows the Cellpose section of `Practical_07_CNN_CVAT_YOLO_Cellpose.ipynb`. Cellpose is used as a **pretrained** instance-segmentation model; we do not train it from scratch.

A bounding box only localises an object. An instance mask gives a pixel-level identity for each cell, which is why area, eccentricity and intensity can be measured afterwards.


## 0. Setup

Inference is much cheaper than YOLO/CNN training, but still faster on GPU. `RUN_EXPENSIVE` gates the Cellpose calls.


In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

import torch

RANDOM_STATE = 221026
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

RUN_EXPENSIVE = True
SHOW_PLOTS = True
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}


def locate_week7_root():
    current = Path.cwd().resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'week7'):
            if (candidate / 'detection').is_dir() and any(
                (candidate / name).is_dir() for name in ('segemntation', 'segmentation')
            ):
                return candidate
    return current


WEEK7_ROOT = locate_week7_root()
SEGMENTATION_ROOT = next(
    (WEEK7_ROOT / name for name in ('segemntation', 'segmentation') if (WEEK7_ROOT / name).is_dir()),
    WEEK7_ROOT / 'segmentation',
)
print('WEEK7_ROOT:', WEEK7_ROOT)
print('SEGMENTATION_ROOT:', SEGMENTATION_ROOT)
print('CUDA available:', torch.cuda.is_available())


## 1. Inspect the microscopy images

The folder name in this course dump is `segemntation` (typo). It contains:

- `brightfield`: transmitted-light images
- `clsm`: confocal laser scanning microscopy

We record format, mode, shape, channel count, dtype and intensity range. These properties decide the Cellpose `channels` argument. For a grayscale image, `channels=[0, 0]` means the same image is used without a separate nuclear channel.


In [ ]:
def image_inventory_row(path, source):
    row = {'source': source, 'file': path.name, 'readable': False}
    try:
        with Image.open(path) as image:
            array = np.asarray(image)
            n_frames = getattr(image, 'n_frames', 1)
            row.update({
                'readable': True,
                'format': image.format,
                'mode': image.mode,
                'width': image.width,
                'height': image.height,
                'n_frames': int(n_frames),
                'array_shape': tuple(array.shape),
                'dtype': str(array.dtype),
                'channels': 1 if array.ndim == 2 else array.shape[-1],
                'min_intensity': float(np.nanmin(array)),
                'max_intensity': float(np.nanmax(array)),
                'finite': bool(np.isfinite(array).all()),
            })
    except Exception as error:
        row['error'] = repr(error)
    return row


brightfield_files = sorted(
    path for path in (SEGMENTATION_ROOT / 'brightfield').glob('*')
    if path.suffix.lower() in IMAGE_EXTENSIONS
) if (SEGMENTATION_ROOT / 'brightfield').is_dir() else []
clsm_files = sorted(
    path for path in (SEGMENTATION_ROOT / 'clsm').glob('*')
    if path.suffix.lower() in IMAGE_EXTENSIONS
) if (SEGMENTATION_ROOT / 'clsm').is_dir() else []

print('Brightfield images:', len(brightfield_files))
print('CLSM images:', len(clsm_files))

inventory_rows = []
for source, files in (('brightfield', brightfield_files), ('clsm', clsm_files)):
    for path in files:
        inventory_rows.append(image_inventory_row(path, source))
image_inventory = pd.DataFrame(inventory_rows)
display(image_inventory)
if len(image_inventory):
    print('Unreadable files:', int((~image_inventory['readable']).sum()))


## 2. Visual preview of both modalities

The same pretrained model can behave differently on brightfield vs confocal data. We therefore look at representative images before choosing diameters and thresholds.


In [ ]:
def show_image_panel(paths, title, n=4):
    if not paths:
        print(f'No images for {title}')
        return
    chosen = paths[:n]
    figure, axes = plt.subplots(1, len(chosen), figsize=(4.2 * len(chosen), 4), squeeze=False)
    for axis, path in zip(axes.ravel(), chosen):
        array = np.asarray(Image.open(path))
        if array.ndim == 2:
            axis.imshow(array, cmap='gray')
        else:
            axis.imshow(array[..., :3] if array.shape[-1] >= 3 else array)
        axis.set_title(path.name, fontsize=8)
        axis.axis('off')
    figure.suptitle(title)
    plt.tight_layout()
    plt.show()


if SHOW_PLOTS:
    show_image_panel(brightfield_files, 'Brightfield preview')
    show_image_panel(clsm_files, 'CLSM preview')


## 3. Cellpose parameters

The values are imaging assumptions, not universal biological truths.

| Parameter | Meaning | Typical effect |
|---|---|---|
| `model` | Pretrained checkpoint | A model trained on a different modality can miss or over-segment cells |
| `flow_threshold` | Maximum allowed flow error for keeping a mask | Lower = stricter; higher = keeps irregular / uncertain masks |
| `cellprob_threshold` | Pixel-level cell-probability cutoff | Lower finds more/larger objects; higher is conservative |
| `min_size` | Minimum object area in pixels | Higher removes fragments; lower keeps small objects |

Homework item 5: compare **at least two** settings. We compare `default` vs `sensitive` vs `conservative` on the **same** images, then decide which mask is biologically more plausible.


In [ ]:
CELLPOSE_PARAMETER_SETS = {
    'default': {
        'model': 'cpsam',
        'flow_threshold': 0.4,
        'cellprob_threshold': 0.0,
        'min_size': 15,
    },
    'sensitive': {
        'model': 'cpsam',
        'flow_threshold': 0.2,
        'cellprob_threshold': -1.0,
        'min_size': 5,
    },
    'conservative': {
        'model': 'cpsam',
        'flow_threshold': 0.8,
        'cellprob_threshold': 0.5,
        'min_size': 30,
    },
}
COMPARE_SETTINGS = ['default', 'sensitive']  # the two settings required by the homework
display(pd.DataFrame(CELLPOSE_PARAMETER_SETS).T)
print('Compared settings:', COMPARE_SETTINGS)

# Representative images: one brightfield and one confocal if both exist.
selected_sources = []
if brightfield_files:
    selected_sources.append(brightfield_files[min(4, len(brightfield_files) - 1)])
if clsm_files:
    selected_sources.append(clsm_files[0])
print('Images used for the comparison:')
for path in selected_sources:
    print('-', path)


## 4. Run Cellpose for each setting

The same image is segmented twice (or more). Only then can we attribute mask differences to the parameter change rather than to a different field of view.


In [ ]:
def load_cellpose_image(path):
    image = Image.open(path)
    if image.mode in {'RGB', 'RGBA', 'P'}:
        array = np.asarray(image.convert('L'), dtype=np.float32)
    else:
        array = np.asarray(image, dtype=np.float32)
        if array.ndim == 3:
            array = array.mean(axis=-1)
    return array


def run_cellpose(image, params):
    from cellpose import models as cellpose_models
    model = cellpose_models.CellposeModel(
        gpu=torch.cuda.is_available(),
        pretrained_model=params['model'],
    )
    result = model.eval(
        image,
        channels=[0, 0],
        diameter=None,
        flow_threshold=params['flow_threshold'],
        cellprob_threshold=params['cellprob_threshold'],
        min_size=params['min_size'],
    )
    return result[0]


segmentation_results = {}
if RUN_EXPENSIVE and selected_sources:
    for image_path in selected_sources:
        image = load_cellpose_image(image_path)
        segmentation_results[image_path.name] = {'path': image_path, 'image': image, 'masks': {}}
        for setting in COMPARE_SETTINGS:
            params = CELLPOSE_PARAMETER_SETS[setting]
            print(f'{image_path.name} | {setting} | {params}')
            masks = run_cellpose(image, params)
            segmentation_results[image_path.name]['masks'][setting] = masks
            print('  objects:', int(masks.max()) if masks is not None else 0)
else:
    print('Cellpose inference skipped. Set RUN_EXPENSIVE=True to run it.')


## 5. Side-by-side masks

Look for merged neighbouring cells, fragments of debris, missed dim cells, and whether the mask follows the actual cell boundary rather than a halo around it.


In [ ]:
if SHOW_PLOTS and segmentation_results:
    for image_name, payload in segmentation_results.items():
        n_panels = 1 + len(payload['masks'])
        figure, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5), squeeze=False)
        axes = axes.ravel()
        axes[0].imshow(payload['image'], cmap='gray')
        axes[0].set_title(f'Input: {image_name}')
        axes[0].axis('off')
        for axis, (setting, masks) in zip(axes[1:], payload['masks'].items()):
            axis.imshow(masks, cmap='tab20')
            n_objects = int(masks.max()) if masks is not None else 0
            axis.set_title(f'{setting}\n{n_objects} objects')
            axis.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('Mask visualisation skipped.')


## 6. From masks to measurements

A usable instance mask has one label per cell, not merged clumps or background fragments. We extract area, centroid, eccentricity, solidity and mean intensity, then compare the two parameter settings.

Suspicious patterns:

- many objects with tiny area → debris / over-segmentation
- a few objects with huge area → merged cells
- eccentricity near 1 for cells that should be round → mask shape is wrong


In [ ]:
from skimage.measure import regionprops_table

measurement_frames = []
if segmentation_results:
    for image_name, payload in segmentation_results.items():
        for setting, masks in payload['masks'].items():
            if masks is None or int(masks.max()) == 0:
                measurement_frames.append(pd.DataFrame({
                    'image': [image_name], 'setting': [setting], 'label': [pd.NA], 'area': [0],
                }))
                continue
            table = pd.DataFrame(regionprops_table(
                masks,
                intensity_image=payload['image'],
                properties=('label', 'area', 'centroid', 'bbox', 'eccentricity', 'solidity', 'mean_intensity'),
            ))
            table['image'] = image_name
            table['setting'] = setting
            measurement_frames.append(table)
    measurements = pd.concat(measurement_frames, ignore_index=True) if measurement_frames else pd.DataFrame()
    display(measurements.head(20))

    if len(measurements):
        summary = (
            measurements.groupby(['image', 'setting'], dropna=False)
            .agg(
                n_objects=('label', 'count'),
                mean_area=('area', 'mean'),
                median_area=('area', 'median'),
                min_area=('area', 'min'),
                max_area=('area', 'max'),
                mean_eccentricity=('eccentricity', 'mean'),
                mean_solidity=('solidity', 'mean'),
            )
            .reset_index()
        )
        display(summary)
        summary.to_csv(WEEK7_ROOT / 'homework' / 'cellpose_metrics.csv', index=False)
        print('Saved:', WEEK7_ROOT / 'homework' / 'cellpose_metrics.csv')

        if SHOW_PLOTS:
            for image_name in measurements['image'].unique():
                subset = measurements[measurements['image'] == image_name]
                figure, axes = plt.subplots(1, 2, figsize=(12, 4))
                for setting, group in subset.groupby('setting'):
                    axes[0].hist(group['area'].dropna(), bins=20, alpha=0.5, label=setting)
                    axes[1].hist(group['eccentricity'].dropna(), bins=20, alpha=0.5, label=setting)
                axes[0].set_title(f'Object area: {image_name}')
                axes[1].set_title(f'Eccentricity: {image_name}')
                axes[0].legend()
                axes[1].legend()
                plt.tight_layout()
                plt.show()
else:
    print('Run Cellpose before extracting morphology.')


## 7. Which setting is biologically more plausible?

Use the masks and the measurement table, not the object count alone.

Typical trade-off:

- **Sensitive** (`lower cellprob_threshold`, smaller `min_size`): more objects, including dim cells, but also fragments and noise.
- **Default / conservative**: fewer fragments, but neighbouring cells may merge and small true cells may disappear.

Write which setting you would keep for this modality and why. If brightfield and CLSM disagree, say so: the same thresholds are not automatically valid across microscopes.


## Report answers (segmentation)

Fill these after the figures above. They map to homework questions 1, 5 and 6.

**1. What image, annotation, and split checks were necessary before modelling?**

- File type, mode, channel count, dtype, intensity range, and whether the image is brightfield or confocal. Cellpose here is pretrained, so there is no train/val split, but the channel argument and assumed cell size still have to be recorded.

**5. Which Cellpose parameters changed the masks most, and what trade-off did they create?**

- Parameters changed: `flow_threshold`, `cellprob_threshold`, `min_size`.
- What changed in the masks: ...
- Trade-off: ...
- More plausible setting: ... because ...

**6. What information would be needed to compare segmentation fairly across biological experiments?**

- Pixel size / magnification, staining, number of channels, a small manually annotated subset for precision/recall of instances, and the exact Cellpose model name plus all four parameters above.
